# 03 — Construcción de OBT (One Big Table)

Ensambla `NYC_TAXI_P3.ANALYTICS.OBT_TRIPS` a partir de `RAW.TRIPS_ENRICHED``.

**Grano:** 1 fila = 1 viaje.  
**Contenido:** hechos + zonas desnormalizadas + catálogos decodificados + derivadas + lineage.

**Derivadas calculadas:**
- `trip_duration_min` = DATEDIFF en minutos entre pickup y dropoff.
- `avg_speed_mph` = trip_distance / (duration_min / 60). NULL si duración o distancia ≤ 0.
- `tip_pct` = (tip_amount / fare_amount) × 100. NULL si fare_amount ≤ 0.

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

print("=" * 60)
print("CELDA 1: Inicialización Spark + Configuración Snowflake")
print("=" * 60)

spark = SparkSession.builder.appName("NYC_Taxi_OBT").getOrCreate()
print("✓ SparkSession creada")

SF_OPTIONS = {
    "sfURL":       f"{os.environ['SF_ACCOUNT']}.snowflakecomputing.com",
    "sfUser":      os.environ["SF_USER"],
    "sfPassword":  os.environ["SF_PASSWORD"],
    "sfDatabase":  os.environ["SF_DATABASE"],
    "sfSchema":    os.environ["SF_SCHEMA_RAW"],
    "sfWarehouse": os.environ["SF_WAREHOUSE"],
    "sfRole":      os.environ["SF_ROLE"],
}

SF_OPTS_ANALYTICS = {**SF_OPTIONS, "sfSchema": os.environ["SF_SCHEMA_ANALYTICS"]}

print(f"✓ Snowflake URL: {SF_OPTIONS['sfURL']}")
print(f"✓ Schema RAW: {os.environ['SF_SCHEMA_RAW']}")
print(f"✓ Schema ANALYTICS: {os.environ['SF_SCHEMA_ANALYTICS']}")
print("=" * 60)

CELDA 1: Inicialización Spark + Configuración Snowflake
✓ SparkSession creada
✓ Snowflake URL: AASKQCL-YY64872.snowflakecomputing.com
✓ Schema RAW: RAW
✓ Schema ANALYTICS: ANALYTICS


## 3.1 Crear OBT con CTAS (CREATE TABLE AS SELECT)

Se ejecuta un `CREATE OR REPLACE TABLE` directamente en Snowflake desde la tabla `TRIPS_ENRICHED`.
Aquí se calculan las derivadas.

In [2]:
from py4j.java_gateway import java_import
java_import(spark._jvm, "net.snowflake.spark.snowflake.Utils")
SfUtils = spark._jvm.net.snowflake.spark.snowflake.Utils

print("=" * 60)
print("CELDA 2: CTAS — Crear OBT_TRIPS en ANALYTICS")
print("=" * 60)

ctas_sql = """
CREATE OR REPLACE TABLE NYC_TAXI_P3.ANALYTICS.OBT_TRIPS AS
SELECT
    pickup_datetime,
    dropoff_datetime,
    TO_DATE(pickup_datetime)  AS pickup_date,
    EXTRACT(HOUR FROM pickup_datetime)  AS pickup_hour,
    TO_DATE(dropoff_datetime) AS dropoff_date,
    EXTRACT(HOUR FROM dropoff_datetime) AS dropoff_hour,
    DAYOFWEEK(pickup_datetime) AS day_of_week,
    source_month AS month,
    source_year AS year,

    pu_location_id,
    pu_zone,
    pu_borough,
    do_location_id,
    do_zone,
    do_borough,

    service_type,
    vendor_id,
    vendor_name,
    rate_code_id,
    rate_code_desc,
    payment_type,
    payment_type_desc,
    trip_type,

    passenger_count,
    trip_distance,
    store_and_fwd_flag,

    fare_amount,
    extra,
    mta_tax,
    tip_amount,
    tolls_amount,
    improvement_surcharge,
    congestion_surcharge,
    airport_fee,
    total_amount,

    -- Derivadas
    DATEDIFF('minute', pickup_datetime, dropoff_datetime) AS trip_duration_min,
    CASE
        WHEN DATEDIFF('minute', pickup_datetime, dropoff_datetime) > 0 AND trip_distance > 0
        THEN ROUND(trip_distance / (DATEDIFF('minute', pickup_datetime, dropoff_datetime) / 60.0), 2)
        ELSE NULL
    END AS avg_speed_mph,
    CASE
        WHEN fare_amount > 0 THEN ROUND((tip_amount / fare_amount) * 100, 2)
        ELSE NULL
    END AS tip_pct,

    -- Lineage
    run_id,
    ingested_at_utc,
    service_type AS source_service,
    source_year,
    source_month

FROM NYC_TAXI_P3.RAW.TRIPS_ENRICHED
"""

print(">>> Ejecutando CTAS en Snowflake (server-side, sin mover datos al driver)...")
SfUtils.runQuery(SF_OPTIONS, ctas_sql)
print("✓ OBT_TRIPS creada exitosamente en ANALYTICS")

# Verificar conteo
count_sql = "SELECT COUNT(*) AS cnt FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS"
df_cnt = (spark.read.format("snowflake")
    .options(**SF_OPTS_ANALYTICS).option("query", count_sql).load())
total = df_cnt.collect()[0][0]
print(f"✓ Total filas en OBT_TRIPS: {total:,}")
print("=" * 60)

CELDA 2: CTAS — Crear OBT_TRIPS en ANALYTICS
>>> Ejecutando CTAS en Snowflake (server-side, sin mover datos al driver)...
✓ OBT_TRIPS creada exitosamente en ANALYTICS
✓ Total filas en OBT_TRIPS: 815,553,850


## 3.2 Verificación de conteo por servicio

In [6]:
print("=" * 60)
print("CELDA 3: Conteo por servicio en OBT_TRIPS")
print("=" * 60)

svc_sql = """
SELECT service_type, COUNT(*) AS total_rows
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
GROUP BY service_type
ORDER BY service_type
"""
df_svc = (spark.read.format("snowflake")
    .options(**SF_OPTS_ANALYTICS).option("query", svc_sql).load())
df_svc.show(truncate=False)
print("=" * 60)

CELDA 3: Conteo por servicio en OBT_TRIPS
+------------+----------+
|SERVICE_TYPE|TOTAL_ROWS|
+------------+----------+
|green       |65590748  |
|yellow      |749963102 |
+------------+----------+



## 3.3 Verificación de idempotencia

Re-ejecutamos la CTAS y verificamos que el conteo no cambia (CREATE OR REPLACE = idempotente).

In [7]:
print("=" * 60)
print("CELDA 4: Verificación de idempotencia")
print("=" * 60)

count_sql = "SELECT COUNT(*) AS cnt FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS"

# Contar filas antes
rows_before = (spark.read.format("snowflake")
    .options(**SF_OPTS_ANALYTICS).option("query", count_sql).load()
    .collect()[0][0])
print(f">>> Filas antes de rebuild: {rows_before:,}")

# Re-ejecutar CTAS
print(">>> Re-ejecutando CTAS (CREATE OR REPLACE)...")
SfUtils.runQuery(SF_OPTIONS, ctas_sql)

# Contar filas después
rows_after = (spark.read.format("snowflake")
    .options(**SF_OPTS_ANALYTICS).option("query", count_sql).load()
    .collect()[0][0])
print(f">>> Filas después de rebuild: {rows_after:,}")

assert rows_before == rows_after, f"FALLO IDEMPOTENCIA: {rows_before} != {rows_after}"
print(f"✓ Idempotencia verificada: {rows_before:,} == {rows_after:,}")
print("=" * 60)

CELDA 4: Verificación de idempotencia
>>> Filas antes de rebuild: 815,553,850
>>> Re-ejecutando CTAS (CREATE OR REPLACE)...
>>> Filas después de rebuild: 815,553,850
✓ Idempotencia verificada: 815,553,850 == 815,553,850


## 3.4 Muestra de la OBT final

In [5]:
print("=" * 60)
print("CELDA 5: Muestra de la OBT final")
print("=" * 60)

sample_sql = "SELECT * FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS LIMIT 10"
print(">>> Consultando muestra...")
df_sample = (spark.read.format("snowflake")
    .options(**SF_OPTS_ANALYTICS).option("query", sample_sql).load())
df_sample.show(5, truncate=False)
print("✓ NB03 completado — OBT construida y verificada")
print("=" * 60)

CELDA 5: Muestra de la OBT final
>>> Consultando muestra...
+-------------------+-------------------+-----------+-----------+------------+------------+-----------+-----+----+--------------+-----------------+----------+--------------+-----------------------------+----------+------------+---------+-------------+------------+---------------+------------+-----------------+---------+---------------+-------------+------------------+-----------+-----+-------+----------+------------+---------------------+--------------------+-----------+------------+-----------------+-------------+-------+----------------+-------------------+--------------+-----------+------------+
|PICKUP_DATETIME    |DROPOFF_DATETIME   |PICKUP_DATE|PICKUP_HOUR|DROPOFF_DATE|DROPOFF_HOUR|DAY_OF_WEEK|MONTH|YEAR|PU_LOCATION_ID|PU_ZONE          |PU_BOROUGH|DO_LOCATION_ID|DO_ZONE                      |DO_BOROUGH|SERVICE_TYPE|VENDOR_ID|VENDOR_NAME  |RATE_CODE_ID|RATE_CODE_DESC |PAYMENT_TYPE|PAYMENT_TYPE_DESC|TRIP_TYPE|PASSENGER_COU